
# Module 22: XGBoost (Practice Notebook)

### Instructions for Students
- This is a **practice notebook**.
- Complete all **TODO** sections.
- Read the markdown explanations carefully.
- Do not skip evaluation and reflection questions.

Dataset used here is **California Housing (Regression)**.



## 1. Import Required Libraries


In [7]:
# TODO: Import necessary libraries

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor


## 2. Load Dataset (California Housing)


In [6]:
from sklearn.datasets import fetch_openml

data = fetch_openml(name="california_housing", version=1, as_frame=True)
X = data.data
y = data.target

# One-hot encode the 'ocean_proximity' column
X = pd.get_dummies(X, columns=['ocean_proximity'], drop_first=True)


## 3. Train-Test Split


In [8]:
# TODO: Perform train-test split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)



## 4. Baseline XGBoost Regressor


In [9]:
# TODO: Train baseline model
baseline_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="logloss",
    random_state=42
)

baseline_model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='logloss',
             feature_types=None, feature_weights=None, gamma=None,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, ...)


## 5. Evaluate Baseline Model


In [10]:
# TODO: Evaluate baseline
# TODO: Evaluate baseline

y_pred_baseline = baseline_model.predict(X_test)

mse_baseline = mean_squared_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)

print("Baseline Model Performance")
print("MSE:", mse_baseline)
print("RMSE:", rmse_baseline)
print("R2 Score:", r2_baseline)

Baseline Model Performance
MSE: 3209475584.0
RMSE: 56652.23370706578
R2 Score: 0.7550783157348633



## 6. Hyperparameter Tuning with GridSearchCV


In [11]:
# TODO: Define parameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0]
}


### Base Model for Grid Search


In [12]:
# TODO: Base model

base_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)



### Run GridSearchCV


In [13]:
# TODO: Run GridSearchCV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="r2",
    cv=3,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)



Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best Parameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}
Best Score: 0.8333459695180258



## 7. Evaluate Tuned Model


In [14]:
# TODO: Evaluate tuned model
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

mse_tuned = mean_squared_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mse_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print("\nTuned Model Performance")
print("MSE:", mse_tuned)
print("RMSE:", rmse_tuned)
print("R2 Score:", r2_tuned)





Tuned Model Performance
MSE: 2200541440.0
RMSE: 46909.92901294991
R2 Score: 0.8320721387863159



# **Early Stopping (Overfitting Control)**
Early stopping halts training when validation performance stops improving.




In [15]:
import xgboost as xgb

# Convert dataset to DMatrix (optimized data structure for XGBoost)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Parameters for regression
params = {
    "objective": "reg:squarederror",   # ✅ regression objective
    "eval_metric": "rmse",             # better for regression
    "max_depth": 3,
    "eta": 0.05,                       # learning_rate
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

# Train with Early Stopping
modelr = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=500,
    evals=[(dtrain, "train"), (dtest, "eval")],
    early_stopping_rounds=20,
    verbose_eval=True
)

[0]	train-rmse:113768.14273	eval-rmse:112631.43945
[1]	train-rmse:110982.42929	eval-rmse:109958.43262
[2]	train-rmse:107973.02585	eval-rmse:107056.19938
[3]	train-rmse:105175.70683	eval-rmse:104345.07713
[4]	train-rmse:102894.75486	eval-rmse:102161.77209
[5]	train-rmse:100728.99761	eval-rmse:100052.29125
[6]	train-rmse:98438.28397	eval-rmse:97852.65696
[7]	train-rmse:96908.76749	eval-rmse:96319.45437
[8]	train-rmse:95635.08064	eval-rmse:95044.69218
[9]	train-rmse:94253.09404	eval-rmse:93667.12909
[10]	train-rmse:93025.69053	eval-rmse:92393.42130
[11]	train-rmse:91299.05823	eval-rmse:90761.24145
[12]	train-rmse:89677.55122	eval-rmse:89232.43495
[13]	train-rmse:88598.93594	eval-rmse:88116.30330
[14]	train-rmse:86985.18240	eval-rmse:86580.84771
[15]	train-rmse:85468.28097	eval-rmse:85164.54219
[16]	train-rmse:84083.21106	eval-rmse:83882.16120
[17]	train-rmse:82821.42717	eval-rmse:82723.88044
[18]	train-rmse:81988.67206	eval-rmse:81885.17003
[19]	train-rmse:80883.51634	eval-rmse:80884.5369

In [16]:
# Predictions
y_pred = modelr.predict(dtest)

# Evaluation
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Early Stopping Model Performance")
print("RMSE:", rmse)
print("R2 Score:", r2)

Early Stopping Model Performance
RMSE: 53387.99385629694
R2 Score: 0.7824894189834595



## 8. Reflection Questions

1. Did GridSearch improve performance?
2. Which parameter had the biggest effect?
3. What happens if learning_rate is too high?
4. Would you deploy this model? Why?


# 🧠 8️⃣ Reflection Questions (Markdown Answers)
**1️⃣ Did GridSearch improve performance?**

Yes, GridSearch typically improves performance by finding optimal hyperparameters. The tuned model usually achieves higher R² and lower RMSE than the baseline model.

**2️⃣ Which parameter had the biggest effect?**

Usually:

learning_rate

max_depth

n_estimators

Learning rate often has the biggest impact because it controls how fast the model learns.

**3️⃣ What happens if learning_rate is too high?**

If learning_rate is too high:

Model may overfit

Training becomes unstable

Performance decreases

Model may miss optimal solution

**4️⃣ Would you deploy this model? Why?**

Yes, if:

R² is high

RMSE is reasonably low

Model generalizes well on unseen data

Before deployment, I would:

Test on new unseen dataset

Monitor performance

Check for overfitting